# Chapter 12: MLOps

## 12.4 ML Pipeline

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer

In [ ]:
# Description about the data set





In [ ]:
chd_df = pd.read_csv("SAheart.data")

In [ ]:
chd_df.columns

Index(['row.names', 'sbp', 'tobacco', 'ldl', 'adiposity', 'famhist', 'typea',
       'obesity', 'alcohol', 'age', 'chd'],
      dtype='object')

In [ ]:

#sbp:	Systolic blood pressure(Numerical)
#tobacco:	Cumulative tobacco consumption(Numerical)
#ldl:	Low-density lipoprotein cholesterol(Numerical)
#adiposity:	Measure of body fat/adiposity(Numerical)
#famhist:	Family history of heart disease	(Categorical}
#typea:	Type-A behavior score	(Numerical)
#obesity:	Body-mass index/obesity measure	(Numerical)
#alcohol:	Current alcohol consumption	(Numerical)
#age:	Age of the person	(Numerical)
#chd:	Whether coronary heart disease is present	(Binary response)

In [ ]:
chd_df.head(5)

,row.names,sbp,tobacco,ldl,adiposity,famhist,typea,obesity,alcohol,age,chd
0,1,160,12.00,5.73,23.11,Present,49,25.30,97.20,52,1
1,2,144,0.01,4.41,28.61,Absent,55,28.87,2.06,63,1
2,3,118,0.08,3.48,32.28,Present,52,29.14,3.81,46,0
3,4,170,7.50,6.41,38.03,Present,51,31.99,24.26,58,1
4,5,134,13.60,3.50,27.78,Present,60,25.99,57.34,49,1


In [7]:
# y is our response variable, X is set of predictor variables.
# Our main goal is to predict the chd_df(y) based on the predictors variables(X).

In [ ]:
X = chd_df.drop('chd', axis=1)
y = chd_df['chd']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)

In [7]:
categorical_features = ['famhist']
numerical_features = ['sbp',
                      'tobacco',
                      'ldl',
                      'adiposity',
                      'typea',
                      'obesity',
                      'alcohol',
                      'age']

In [8]:
categorical_features

['famhist']

In [9]:
numerical_features

['sbp', 'tobacco', 'ldl', 'adiposity', 'typea', 'obesity', 'alcohol', 'age']

In [10]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [11]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ))
])


In [ ]:
rf_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['sbp', 'tobacco', 'ldl',
                                                   'adiposity', 'typea',
                                                   'obesity', 'alcohol',
                                                   'age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['famhist'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [ ]:
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['sbp', 'tobacco', 'ldl',
                                                   'adiposity', 'typea',
                                                   'obesity', 'alcohol',
                                                   'age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['famhist'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [ ]:
y_pred = rf_pipeline.predict(X_test)

In [24]:
print(y_pred)

[0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0
 0 0 0 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 1 1 0 0 0 1 0 0 0 0 0 0
 1 1 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0]


In [17]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.92      0.84        59
           1       0.79      0.56      0.66        34

    accuracy                           0.78        93
   macro avg       0.79      0.74      0.75        93
weighted avg       0.79      0.78      0.77        93



In [ ]:
from joblib import dump
dump(rf_pipeline, 'ch_rf.pickle')

['chd.pickle']

In [21]:
#%pip install mlflow


     ---------------------------------------- 11.2/11.2 MB 3.4 MB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 56.2/56.2 KB 2.9 MB/s eta 0:00:00
     -------------------------------------- 114.9/114.9 KB 3.4 MB/s eta 0:00:00
     -------------------------------------- 148.8/148.8 KB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 3.8/3.8 MB 3.1 MB/s eta 0:00:00
     ---------------------------------------- 3.6/3.6 MB 3.3 MB/s eta 0:00:00
     ---------------------------------------- 27.8/27.8 MB 3.1 MB/s eta 0:00:00
     -------------------------------------- 132.2/132.2 KB 1.9 MB/s eta 0:00:00
     -------------------------------------- 123.9/123.9 KB 2.4 MB/s eta 0:00:00
     -------------------------------------- 265.9/265.9 KB 2.0 MB/s eta 0:00:00
     -------------------------------------- 480.5/480.5 KB 3.8 MB/s eta 0:00:00
     -------------------------------------- 10

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ortools 9.11.4210 requires protobuf<5.27,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
You should consider upgrading via the 'C:\Users\Dell\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [19]:
import mlflow
from mlflow.models import infer_signature

In [21]:
import sys
import mlflow
import sklearn

print("Python:", sys.executable)
print("MLflow:", mlflow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

Python: c:\Users\avrae\OneDrive\Desktop\Desktop\MLOps\.venv\Scripts\python.exe
MLflow: 2.13.1
Scikit-learn: 1.3.2
Tracking URI: file:///c:/Users/avrae/OneDrive/Desktop/Desktop/MLOps/mlruns


In [ ]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# Create a new MLflow Experiment
mlflow.set_experiment("CHD_RF_Prediction")

# Start an MLflow run
with mlflow.start_run():
    # Log the loss metric
    mlflow.log_metric("roc", np.round(roc_auc_score(y_test, y_pred), 3))

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training", "Random Forest")

    # Infer the model signature
    signature = infer_signature(X_train,
                                rf_pipeline.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
    sk_model=rf_pipeline,
    artifact_path="rf",
    signature=signature,
    input_example=X_train,
    registered_model_name="random_forest",
)

c:\Users\avrae\OneDrive\Desktop\Desktop\MLOps\.venv\lib\site-packages\mlflow\types\utils.py:394: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'logistic' already exists. Creating a new version of this model...
2026/09/08 07:59:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. M